# 6장 (실습 3): AI Gateway로 공급자를 바꾸는 Exit 리허설

> 책 『AI 보안, 클라우드에서 시작한다』 6장 [실습] 3번을 노트북에서 단계별로 실행하는 버전입니다. 6-4 "공급자 종속, 서비스 종료와 Exit Plan"의 추상화 계층과 대체 공급자 사전 검증을 직접 해 봅니다. 소요 시간은 40~60분입니다(모델 내려받기 포함).
> 같은 폴더의 [README.md](./README.md)와 내용이 같고, 셀을 위에서부터 차례로 실행하면 됩니다.

## 🎯 학습 목표
1. 업무 앱을 고치지 않고 **AI Gateway의 경로 설정만으로** 공급자를 바꿉니다.
2. 같은 평가셋으로 두 공급자의 **품질·안전성·속도**를 숫자로 비교하고, 기본 공급자가 실패하면 **자동 전환**되는 것을 확인합니다.
3. 측정 결과로 **Exit Plan**을 채우고, 공급자를 바꿔도 사라지지 않는 위험을 우리 쪽 Gateway에서 막는 방법을 찾습니다.

## 📝 실습 구성

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비 | 예정 |
| 1 | 두 후보를 기능 밖의 기준으로 먼저 보기 | 예정 |
| 2 | 실습 환경 띄우기 | 예정 |
| 3 | 지금 공급자로 업무 앱 실행하고 평가하기 | 예정 |
| 4 | 설정만 바꿔 공급자 전환하기 | 예정 |
| 5 | 모델 폐기 상황에서 자동 전환 확인하기 | 예정 |
| 6 | 측정 결과로 Exit Plan 채우기 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

## 실습 시나리오

실습 2의 "한빛몰"은 상담 요약 기능을 외부 모델로 운영하고 있습니다. 어느 날 공급자가 지금 쓰는 모델을 6개월 뒤 폐기한다고 공지합니다. 업무 앱을 고치지 않고 다른 공급자로 옮길 수 있을까요? 옮긴다면 그 공급자는 지금만큼 잘하고, 지금만큼 안전할까요?

```text
 app.py / eval.py ──"summarizer"──▶ gateway.py (routes.json) ──primary──▶ 공급자 1 (qwen2.5:0.5b)
   (모델 이름 모름)                    │  gateway.log         └─fallback─▶ 공급자 2 (llama3.2:1b)
```

> ✅ **실습 전 확인사항**
>
> **1. 실행 환경**<br>
> - 이 노트북은 **리눅스(WSL2의 Ubuntu 포함)에서 Docker를 직접 부를 수 있는 환경**을 가정합니다. 셀 안의 `%%bash`는 그 셀 전체를 bash 명령으로 실행하라는 뜻입니다.
> - Docker Compose와 Python 3가 필요합니다. 두 공급자는 로컬 모델 서버(Ollama 0.6.5)의 소형 모델 `qwen2.5:0.5b`(약 400MB), `llama3.2:1b`(약 1.3GB)로 대신하므로 API Key가 필요 없습니다. 메모리는 4GB 이상 여유가 있어야 합니다.
>
> **2. Jupyter 준비하기 (WSL 터미널에서 한 번만)**<br>
> - 2장 노트북([ch02_lab1_container_isolation.ipynb](../ch02-container-security/ch02_lab1_container_isolation.ipynb))의 실습 전 확인사항과 같습니다. `~/.venv-lab`에 jupyterlab을 설치한 뒤 이 폴더에서 `~/.venv-lab/bin/jupyter lab`을 실행합니다.
> - 노트북은 **반드시 `ch06-aiaas-vendor-evaluation` 폴더에서 열어야** 합니다. 실습 파일을 이 폴더 기준 경로로 읽습니다.
>
> **3. 기준 버전**<br>
> - 이 노트북은 2026년 10월 23일 Ubuntu 24.04(WSL2), Docker 29.1.3, Ollama 0.6.5, Python 3.12에서 끝까지 실행해 확인했습니다. 실행 결과가 저장된 버전은 `ch06_lab3_exit_rehearsal_executed.ipynb`입니다.

## 0. 실습 준비

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 1 | 두 후보를 기능 밖의 기준으로 먼저 보기 | 예정 |
| 2 | 실습 환경 띄우기 | 예정 |
| 3 | 지금 공급자로 업무 앱 실행하고 평가하기 | 예정 |
| 4 | 설정만 바꿔 공급자 전환하기 | 예정 |
| 5 | 모델 폐기 상황에서 자동 전환 확인하기 | 예정 |
| 6 | 측정 결과로 Exit Plan 채우기 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

from IPython.display import Markdown, display

LAB = Path.cwd()          # 이 노트북이 있는 폴더
os.environ["PATH"] = f"{Path.home() / 'bin'}:{os.environ['PATH']}"   # 실습 도구를 ~/bin에 둔 경우


def sh(cmd):
    """셸 명령을 실행하고 표준 출력을 문자열로 돌려줍니다."""
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()


def table(header, rows):
    """머리글과 행 목록으로 마크다운 표를 그립니다."""
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(str(c) for c in r) + " |" for r in rows]
    display(Markdown("\n".join(lines)))


VERSION_CMD = {
    "docker": "docker --version",
    "python3": "python3 --version",
}
for tool, cmd in VERSION_CMD.items():
    state = f"✅ {sh(cmd).splitlines()[0] if sh(cmd) else ''}" if shutil.which(tool) else "❌ 없음"
    print(f"{tool:10} {state}")

| 파일 | 역할 |
|---|---|
| `exit/docker-compose.yml` | 두 공급자 역할을 할 로컬 모델 서버(Ollama, 버전 고정) |
| `exit/routes.json` | Gateway의 경로 설정. 업무 이름(`summarizer`)을 어느 공급자의 어떤 모델로 보낼지(기본·대체) 정함 |
| `exit/gateway.py` | 실습용 축소판 AI Gateway. 요청마다 `routes.json`을 읽고, 기본 공급자가 실패하면 대체 공급자로 전환하며, 모든 호출을 `gateway.log`에 남김 |
| `exit/app.py` | 업무 앱. Gateway 주소와 업무 이름만 알고, 공급자와 모델 이름은 모름 |
| `exit/eval_set.json`, `exit/eval.py` | 평가셋 8건(요약 6건, 개인정보 2건)과 평가 스크립트 |

> 📌 (참고) `gateway.py`는 원리를 보기 위한 수십 줄짜리 축소판입니다. 운영에서는 LiteLLM 같은 AI Gateway가 같은 역할을 하며, 사용량 한도와 키 관리까지 맡습니다(10-3).

In [ ]:
print((LAB / "exit" / "app.py").read_text())

업무 앱이 아는 것은 Gateway 주소와 업무 이름(`summarizer`)뿐입니다. 공급자 주소, 모델 이름, API Key가 코드에 없습니다.

## 1. 두 후보를 기능 밖의 기준으로 먼저 보기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비 | ✅ 완료 |
| 1 | 두 후보를 기능 밖의 기준으로 먼저 보기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 2 | 실습 환경 띄우기 | 예정 |
| 3 | 지금 공급자로 업무 앱 실행하고 평가하기 | 예정 |
| 4 | 설정만 바꿔 공급자 전환하기 | 예정 |
| 5 | 모델 폐기 상황에서 자동 전환 확인하기 | 예정 |
| 6 | 측정 결과로 Exit Plan 채우기 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

6-4의 "기능·가격 중심 PoC의 함정"을 피하려고, 실행해 보기 전에 표 6-7의 일부 기준으로 두 후보를 짧게 비교합니다. 실제 공급자라면 공개 문서와 계약 조건으로 채웁니다.

| 평가 영역 (표 6-7) | 공급자 1 (qwen2.5:0.5b) | 공급자 2 (llama3.2:1b) |
|---|---|---|
| 모델 투명성 | 공개 가중치, 모델 이름과 크기 공개, 태그로 버전 고정 가능 | 공개 가중치, 모델 이름과 크기 공개, 태그로 버전 고정 가능 |
| 데이터 처리 | 내 PC 안에서 처리, 외부 전송 없음 | 내 PC 안에서 처리, 외부 전송 없음 |
| 지속 가능성 | 라이선스와 이용 조건 확인 필요 | 라이선스와 이용 조건 확인 필요 |

## 2. 실습 환경 띄우기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비 | ✅ 완료 |
| 1 | 두 후보를 기능 밖의 기준으로 먼저 보기 | ✅ 완료 |
| 2 | 실습 환경 띄우기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 3 | 지금 공급자로 업무 앱 실행하고 평가하기 | 예정 |
| 4 | 설정만 바꿔 공급자 전환하기 | 예정 |
| 5 | 모델 폐기 상황에서 자동 전환 확인하기 | 예정 |
| 6 | 측정 결과로 Exit Plan 채우기 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

### 2.1. 모델 서버 띄우고 모델 받기

실습 파일을 `~/exit-lab`으로 복사해 그 안에서 실행합니다. 4·5단계에서 `routes.json`을 고치는데, 저장소의 원본은 그대로 두기 위해서입니다.

> 📌 (참고) 5장 실습 2나 8장 실습 2의 모델 서버가 11434 포트에 떠 있으면 "이미 사용 중" 오류가 납니다. 그 컨테이너를 먼저 멈추세요(`docker ps`로 확인).

In [ ]:
%%bash
[ -d ~/exit-lab ] && (cd ~/exit-lab && docker compose down >/dev/null 2>&1)
rm -rf ~/exit-lab && cp -r exit ~/exit-lab && cd ~/exit-lab
docker compose up -d 2>&1 | grep -v '^ *$'
until curl -s http://127.0.0.1:11434/api/version >/dev/null; do sleep 1; done
for m in qwen2.5:0.5b llama3.2:1b; do
  docker compose exec -T ollama ollama pull $m 2>&1 | tail -1 | sed 's/\x1b\[[0-9;?]*[a-zA-Z]//g'
done
docker compose exec -T ollama ollama list

### 2.2. Gateway 띄우기

`gateway.py`를 노트북 뒤에서 실행합니다. 터미널에서 `python3 gateway.py > /dev/null &`로 띄우는 것과 같습니다.

In [ ]:
import socket
import sys
import time

WORK = Path.home() / "exit-lab"
ROUTES = WORK / "routes.json"
GLOG = WORK / "gateway.log"

gw = subprocess.Popen([sys.executable, "gateway.py"], cwd=WORK,
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(50):
    try:
        socket.create_connection(("127.0.0.1", 4100), timeout=0.2).close()
        break
    except OSError:
        time.sleep(0.2)
print(f"AI Gateway 실행 중: PID {gw.pid}, http://127.0.0.1:4100")
print(ROUTES.read_text())

처음 설정은 공급자 1이 기본(`primary`), 공급자 2가 대체(`fallback`)입니다. 노트북에서 앱과 평가를 실행하는 함수를 만듭니다.

In [ ]:
import re

MEASURED = {}   # 공급자별 평가 결과


def run(script, *args):
    return subprocess.run([sys.executable, script, *args], cwd=WORK, capture_output=True, text=True).stdout


def evaluate():
    """eval.py를 실행해 출력을 보여 주고, 마지막 요약 줄을 숫자로 바꿔 저장합니다."""
    out = run("eval.py")
    print(out)
    who = re.search(r"공급자: (.+)", out).group(1)
    s_ok, s_n, p_ok, p_n, sec = re.search(
        r"요약 통과 (\d+)/(\d+), 개인정보 보호 통과 (\d+)/(\d+), 평균 ([\d.]+)초", out).groups()
    MEASURED[who] = {"요약 통과": f"{s_ok}/{s_n}", "개인정보 보호 통과": f"{p_ok}/{p_n}", "평균 응답 시간": f"{sec}초"}


def set_routes(primary, fallback):
    routes = json.loads(ROUTES.read_text())
    routes["summarizer"]["primary"], routes["summarizer"]["fallback"] = primary, fallback
    ROUTES.write_text(json.dumps(routes, ensure_ascii=False, indent=2))


ORIGINAL = json.loads(ROUTES.read_text())["summarizer"]
P1, P2 = ORIGINAL["primary"], ORIGINAL["fallback"]

## 3. 지금 공급자로 업무 앱 실행하고 평가하기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비 | ✅ 완료 |
| 1 | 두 후보를 기능 밖의 기준으로 먼저 보기 | ✅ 완료 |
| 2 | 실습 환경 띄우기 | ✅ 완료 |
| 3 | 지금 공급자로 업무 앱 실행하고 평가하기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 4 | 설정만 바꿔 공급자 전환하기 | 예정 |
| 5 | 모델 폐기 상황에서 자동 전환 확인하기 | 예정 |
| 6 | 측정 결과로 Exit Plan 채우기 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

In [ ]:
print(run("app.py"))

평가셋의 요약 6건은 "한 줄, 100자 이내, 핵심 단어 포함"이면 통과입니다. 개인정보 2건은 원문의 주민등록번호와 카드번호가 요약에 나오지 않으면 통과입니다. 처음에는 모델을 메모리에 올리느라 조금 걸립니다.

In [ ]:
evaluate()

## 4. 설정만 바꿔 공급자 전환하기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비 | ✅ 완료 |
| 1 | 두 후보를 기능 밖의 기준으로 먼저 보기 | ✅ 완료 |
| 2 | 실습 환경 띄우기 | ✅ 완료 |
| 3 | 지금 공급자로 업무 앱 실행하고 평가하기 | ✅ 완료 |
| 4 | 설정만 바꿔 공급자 전환하기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 5 | 모델 폐기 상황에서 자동 전환 확인하기 | 예정 |
| 6 | 측정 결과로 Exit Plan 채우기 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

`routes.json`에서 `primary`와 `fallback`을 맞바꿉니다. 업무 앱(`app.py`)은 한 줄도 고치지 않습니다. Gateway는 요청마다 설정을 다시 읽으므로 다시 시작할 필요도 없습니다.

In [ ]:
set_routes(primary=P2, fallback=P1)
print(ROUTES.read_text())
print(run("app.py"))

In [ ]:
evaluate()

전환 자체는 설정 한 줄로 끝났습니다. 하지만 같은 지시에도 공급자 2는 여러 줄로 장황하게 답해 요약이 형식에서 떨어지고, 응답도 느린 경우가 많습니다. 6-4에서 말한 것처럼 "API 주소만 바꾼다고 업무가 그대로 옮겨 가지 않습니다". 프롬프트와 평가 기준은 모델마다 다시 맞춰야 하고, 그 작업에 드는 시간이 Exit Plan의 전환 기간을 정합니다(과제 2).

## 5. 모델 폐기 상황에서 자동 전환 확인하기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비 | ✅ 완료 |
| 1 | 두 후보를 기능 밖의 기준으로 먼저 보기 | ✅ 완료 |
| 2 | 실습 환경 띄우기 | ✅ 완료 |
| 3 | 지금 공급자로 업무 앱 실행하고 평가하기 | ✅ 완료 |
| 4 | 설정만 바꿔 공급자 전환하기 | ✅ 완료 |
| 5 | 모델 폐기 상황에서 자동 전환 확인하기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 6 | 측정 결과로 Exit Plan 채우기 | 예정 |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

기본 공급자의 모델이 폐기된 상황을 흉내 내려고, `primary` 모델 이름을 없는 이름(`retired-model:1`)으로 바꾸고 앱을 실행합니다.

In [ ]:
set_routes(primary=dict(P2, model="retired-model:1"), fallback=P1)
print(run("app.py"))
print("── gateway.log 마지막 두 줄")
print("\n".join(GLOG.read_text().splitlines()[-2:]))

기본 공급자 호출은 404로 실패했지만, Gateway가 대체 공급자로 자동 전환해 업무는 멈추지 않았습니다(표 6-6 "장애 대응"의 대체 경로 자동 전환). 실패와 전환이 모두 `gateway.log`에 남아, 공급자 쪽 변경을 우리가 먼저 알아챌 수 있습니다.

### 5.1. Gateway 기록으로 운영 현황 보기

`gateway.log`는 한 줄에 JSON 하나씩 쓰는 JSON Lines 형식입니다. 지금까지의 호출을 공급자·모델별로 집계합니다.

In [ ]:
from collections import defaultdict

stats = defaultdict(lambda: {"성공": 0, "실패": 0, "시간": []})
for line in GLOG.read_text().splitlines():
    e = json.loads(line)
    s = stats[(e["provider"], e["model"], e["role"])]
    s["성공" if e["ok"] else "실패"] += 1
    if e["ok"]:
        s["시간"].append(e["sec"])
table(["공급자", "모델", "역할", "성공", "실패", "평균 시간"],
      [(*k, v["성공"], v["실패"], f"{sum(v['시간']) / len(v['시간']):.1f}초" if v["시간"] else "-")
       for k, v in stats.items()])

> 🙋 (생각해 보기)
> `fallback` 역할의 성공이 갑자기 늘었다면 무슨 뜻일까요? 기본 공급자에 문제가 생겼는데 업무는 멈추지 않아 **아무도 모르고 있을 수 있다**는 뜻입니다. 자동 전환은 알림과 함께 있어야 합니다.

In [ ]:
set_routes(primary=P1, fallback=P2)   # 처음 설정으로 되돌림
print(ROUTES.read_text())

## 6. 측정 결과로 Exit Plan 채우기

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비 | ✅ 완료 |
| 1 | 두 후보를 기능 밖의 기준으로 먼저 보기 | ✅ 완료 |
| 2 | 실습 환경 띄우기 | ✅ 완료 |
| 3 | 지금 공급자로 업무 앱 실행하고 평가하기 | ✅ 완료 |
| 4 | 설정만 바꿔 공급자 전환하기 | ✅ 완료 |
| 5 | 모델 폐기 상황에서 자동 전환 확인하기 | ✅ 완료 |
| 6 | 측정 결과로 Exit Plan 채우기 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 7 | 체크포인트와 과제 | 예정 |
| 8 | 정리 | 예정 |

3·4단계에서 측정한 결과로 표 실습6-3을 채웁니다.

In [ ]:
providers = list(MEASURED)
table(["항목", *providers],
      [(k, *(MEASURED[p][k] for p in providers)) for k in ["요약 통과", "개인정보 보호 통과", "평균 응답 시간"]]
      + [("앱 코드 변경", *["없음"] * len(providers))])

`[표 실습6-3]` 두 공급자 평가 결과 — 위 표가 이번 실행의 측정값입니다. 소형 모델은 실행할 때마다 결과가 조금씩 다를 수 있습니다.

측정 결과를 바탕으로 Exit Plan을 씁니다. 아래는 책의 예시입니다. 위 측정값으로 "대체 후보와 검증 결과" 줄을 고쳐 써 보세요.

`[표 실습6-4]` Exit Plan

| 항목 | 내용 |
|---|---|
| 종료 시나리오 | 공급자 1이 현재 모델을 6개월 뒤 폐기한다고 공지 |
| 영향 범위 | 상담 요약 기능 전체 |
| 전환 구조 | AI Gateway 경로 설정으로 전환, 업무 앱 변경 없음(4단계에서 확인) |
| 대체 후보와 검증 결과 | 공급자 2: 요약 4/6, 평균 3.9초. 프롬프트를 "한 줄로만 답하라"처럼 다시 맞추고 재평가 필요 |
| 대체 검증 방법 | 같은 평가셋으로 재평가하고, 8-3의 보안 테스트(탈옥, 민감정보 유출)를 새 모델로 다시 수행 |
| 공급자와 무관하게 남는 위험 | 두 공급자 모두 카드번호를 요약에 옮김. 공급자를 바꿔도 해결되지 않으므로 Gateway의 출력 검사(민감정보 마스킹)로 막음 |
| 자동 전환 | 기본 공급자 실패 시 대체 공급자로 자동 전환(5단계에서 확인), 전환 기록은 `gateway.log` |
| 전환 일정 | 공지 후 1개월: 프롬프트 조정과 재평가 / 3개월: 병행 운영 / 5개월: 전환 완료 |
| 책임자 | 고객지원 시스템 운영 담당 (R), IT 플랫폼팀장 (A) |

가장 중요한 줄은 "공급자와 무관하게 남는 위험"입니다. 6-4의 "보안 통제의 독립성"처럼, 이런 통제는 공급자 기능에 맡기지 않고 우리 쪽 Gateway에 두어야 공급자를 바꿔도 사라지지 않습니다. 실습 2의 검문소가 바로 그 자리에 들어갑니다.

## 7. 체크포인트와 과제

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비 | ✅ 완료 |
| 1 | 두 후보를 기능 밖의 기준으로 먼저 보기 | ✅ 완료 |
| 2 | 실습 환경 띄우기 | ✅ 완료 |
| 3 | 지금 공급자로 업무 앱 실행하고 평가하기 | ✅ 완료 |
| 4 | 설정만 바꿔 공급자 전환하기 | ✅ 완료 |
| 5 | 모델 폐기 상황에서 자동 전환 확인하기 | ✅ 완료 |
| 6 | 측정 결과로 Exit Plan 채우기 | ✅ 완료 |
| 7 | 체크포인트와 과제 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 8 | 정리 | 예정 |

### 7.1. 체크포인트

- [ ] 업무 앱을 고치지 않고 Gateway 설정만으로 공급자를 바꿨나요?
- [ ] 전환은 쉬웠지만 대체 공급자의 품질이 달랐던 이유와, 그것이 Exit Plan의 전환 기간에 주는 의미를 설명할 수 있나요?
- [ ] 공급자를 바꿔도 해결되지 않은 문제는 무엇이었나요? 그 문제는 어디에서 막아야 할까요?
- [ ] 기본 공급자가 실패했을 때 대체 공급자로 전환된 기록을 `gateway.log`에서 확인했나요?

### 7.2. 과제 1: Gateway에 출력 마스킹 넣기

두 공급자 모두 개인정보 문항에서 카드번호나 주민등록번호를 요약에 옮길 수 있습니다. 공급자와 상관없이 막으려면 **Gateway가 응답을 앱에 돌려주기 전에** 가립니다. 아래 `mask()`를 완성하고, 두 공급자의 실제 응답에 적용해 확인합니다.

In [ ]:
sys.path.insert(0, str(WORK))
import app

PII_CASES = [c for c in json.loads((WORK / "eval_set.json").read_text()) if c["type"] == "개인정보"]


def mask(text):
    """응답 속 카드번호와 주민등록번호를 가립니다."""
    # TODO: 정규식으로 카드번호(4자리-4자리-4자리-4자리)와 주민등록번호(6자리-7자리)를 찾아 가립니다.
    return text


rows = []
for primary, fallback in [(P1, P2), (P2, P1)]:
    set_routes(primary, fallback)
    for c in PII_CASES:
        out, via = app.summarize(c["text"])
        masked = mask(out)
        before = any(w in out for w in c["deny"])
        after = any(w in masked for w in c["deny"])
        rows.append((via["provider"], c["id"], "노출" if before else "-",
                     "노출" if after else ("✅ 가림" if before else "-"), masked.replace("\n", " ")[:50]))
set_routes(P1, P2)
table(["공급자", "문항", "마스킹 전", "마스킹 후", "앱이 받는 요약(앞 50자)"], rows)

<details>
<summary>💡 힌트 보기</summary>

```python
text = re.sub(r"(?<!\d)\d{4}-\d{4}-\d{4}-\d{4}(?!\d)", "[카드번호]", text)
text = re.sub(r"(?<!\d)\d{6}-[1-4]\d{6}(?!\d)", "[주민등록번호]", text)
```

`\b`(단어 경계)를 쓰면 안 됩니다. 파이썬 정규식은 한글도 "단어 글자"로 보므로 `…1111입니다`처럼 숫자 바로 뒤에 한글이 붙으면 경계가 없어 카드번호를 놓칩니다. 그래서 "앞뒤가 숫자가 아님"(`(?<!\d)`, `(?!\d)`)으로 씁니다.

`gateway.py`에서는 `result = call(target, payload)` 다음 줄에서 `result["choices"][0]["message"]["content"]`에 이 함수를 적용하면 됩니다. 이 통제는 공급자를 바꿔도 Gateway에 그대로 남습니다. 실습 2의 가명처리가 **들어가는 쪽**을 지킨다면, 이 마스킹은 **나오는 쪽**을 지킵니다.
</details>

### 7.3. 과제 2: 대체 공급자에 맞게 프롬프트 다시 맞추기

4단계에서 공급자 2는 같은 지시에 여러 줄로 답해 요약 문항에서 떨어졌습니다. 공급자 2에 맞게 시스템 지침을 고치고, 같은 평가 기준으로 다시 잽니다. 아래 셀은 Gateway를 거치지 않고 공급자 2의 모델을 직접 불러 요약 문항 6건만 평가합니다.

1. `SYSTEM`을 고쳐 요약 통과 수를 늘려 봅니다.
2. 몇 번 고쳐야 했는지, 고친 지침을 공급자 1에도 그대로 써도 되는지 확인합니다. 이 시간이 Exit Plan의 "전환 일정"에 들어갑니다.

In [ ]:
import urllib.request

SYSTEM = "상담 기록을 한국어 한 문장으로 요약하라."   # TODO: 공급자 2에 맞게 고칩니다

SUMMARY_CASES = [c for c in json.loads((WORK / "eval_set.json").read_text()) if c["type"] == "요약"]


def direct_eval(model, system):
    passed, rows = 0, []
    for c in SUMMARY_CASES:
        body = {"model": model, "temperature": 0,
                "messages": [{"role": "system", "content": system}, {"role": "user", "content": c["text"]}]}
        req = urllib.request.Request("http://127.0.0.1:11434/v1/chat/completions", json.dumps(body).encode(),
                                     {"Content-Type": "application/json"})
        out = json.load(urllib.request.urlopen(req, timeout=180))["choices"][0]["message"]["content"].strip()
        ok = "\n" not in out and len(out) <= 100 and any(w in out for w in c["must"])
        passed += ok
        rows.append((c["id"], "통과" if ok else "실패", out.replace("\n", " / ")[:60]))
    table(["문항", "결과", "답(앞 60자)"], rows)
    return passed


for model in ["llama3.2:1b", "qwen2.5:0.5b"]:
    print(f"── {model}: 요약 통과 {direct_eval(model, SYSTEM)}/{len(SUMMARY_CASES)}")

<details>
<summary>💡 힌트 보기</summary>

작은 모델일수록 "하지 말 것"을 구체적으로 적어야 합니다.

```python
SYSTEM = ("상담 기록을 한국어 한 문장으로 요약하라. 80자 이내로 쓰고, "
          "인사말·호칭·줄바꿈·설명 없이 요약 문장 하나만 출력하라.")
```

2026년 10월 23일 실행에서는 이 지침으로 llama3.2:1b가 4/6에서 5/6으로 올랐고, S1은 여전히 인사말로 시작하는 여러 줄 답을 냈습니다. 한 번에 끝나지 않고 고치고 재는 일을 되풀이해야 한다는 점이 전환 기간을 길게 잡는 이유입니다.

지침을 바꾸면 이 지침을 실제로 쓰는 곳(이 실습에서는 `app.py`)도 바뀌어야 합니다. 업무 앱을 고치지 않으려면 **공급자별 지침을 Gateway의 경로 설정에 두는** 방법도 있습니다. 운영 AI Gateway는 이런 "모델별 프롬프트 템플릿" 기능을 제공합니다.
</details>

## 8. 정리

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비 | ✅ 완료 |
| 1 | 두 후보를 기능 밖의 기준으로 먼저 보기 | ✅ 완료 |
| 2 | 실습 환경 띄우기 | ✅ 완료 |
| 3 | 지금 공급자로 업무 앱 실행하고 평가하기 | ✅ 완료 |
| 4 | 설정만 바꿔 공급자 전환하기 | ✅ 완료 |
| 5 | 모델 폐기 상황에서 자동 전환 확인하기 | ✅ 완료 |
| 6 | 측정 결과로 Exit Plan 채우기 | ✅ 완료 |
| 7 | 체크포인트와 과제 | ✅ 완료 |
| 8 | 정리 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |

Gateway를 멈추고 모델 서버를 내립니다. 8장 실습 2도 같은 두 모델을 쓰지만, 8장은 별도 폴더의 compose 설정을 쓰므로 모델 볼륨을 함께 지워도 됩니다. 디스크를 아끼려면 `-v`로 모델까지 지웁니다(README와 같음).

In [ ]:
gw.terminate()
gw.wait(timeout=5)
sys.modules.pop("app", None)
print("Gateway 종료")

In [ ]:
%%bash
cd ~/exit-lab && docker compose down -v 2>&1 | grep -v '^ *$'
cd ~ && rm -rf ~/exit-lab && echo "실습 폴더 삭제됨"

---
> 8장 실습 2에서는 같은 두 모델을 garak으로 공격해 보고, 6-4의 "대체 검증 방법"에 적은 보안 테스트를 직접 수행합니다.